# CrackLens reproducible report

This notebook runs the complete offline-capable pipeline: deterministic dataset preparation, segmentation training, crack geometry, orientation classification, and code screening.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "src"))
import json
import matplotlib.pyplot as plt
from PIL import Image
from cracklens.data.prepare import prepare_dataset
from cracklens.models.train import train_segmentation
from cracklens.pipeline import analyze_image


In [ ]:
summary = prepare_dataset("data/processed", count=24, seed=42, allow_network=False)
summary.__dict__

In [ ]:
training = train_segmentation("data/processed", "models", epochs=1)
training["train"], training["val"], training["test"]

In [ ]:
from cracklens.models.train import iter_pairs, read_gray, predict_classical, _scores
import pandas as pd
model_config = json.loads(Path("models/cracklens_classical.json").read_text())
error_rows = []
for image_path, mask_path in iter_pairs(Path("data/processed"), "test"):
    pred = predict_classical(read_gray(image_path), model_config)
    truth = read_gray(mask_path) > 0
    error_rows.append({"image": image_path.name, **_scores(pred, truth)})
error_table = pd.DataFrame(error_rows).sort_values("iou")
error_table

In [ ]:
sample = Path("data/processed/images/test").glob("*.png").__next__()
result = analyze_image(sample, model_path="models/cracklens_classical.json", reference_mm=100.0)
result["summary"], result["records"]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(Image.open(sample)); axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(result["overlay"]); axes[1].set_title("CrackLens overlay"); axes[1].axis("off")
plt.tight_layout()

## Summary / ?????

**English.** CrackLens produced a reproducible CPU baseline from paired images and masks. The report records the selected grayscale threshold, split-level IoU/F1 metrics, crack length and width estimates, principal-axis angle, flexural/shear screening class, and the configured serviceability verdict. The generated fallback data are a software smoke test; field claims require a calibrated ruler, a public labelled dataset, and validation against engineer-reviewed measurements.

**?????.** CrackLens ?? ?? ???? ???? ????? ???? ?????????? ??????????? ????? ???? ????? ?????? ??? ???? ?? ???? ? ???????? ??? ??? ????? ??????. ????? ??? ???? ????? ?? ???? ?????? ????????? ???? ????? ????????? ????? ? ???? ?????????? ?????? ???? ??????? ???? ??????? ?????? ???? ????? ?????? ???????? ???? ??????????????? ????? ? ??????? ????? ???? ????? ???.